In [ ]:
# ==========================================
# CELL 1: SYSTEM & LIBRARY INSTALLATION
# ==========================================

# 1. Install system-level packages using apt-get (Linux command)
# tesseract-ocr: The actual underlying engine that reads text from images.
# poppler-utils: A system library required to convert PDF pages into raw images.
!apt-get update -qq
!apt-get install -y tesseract-ocr poppler-utils -qq

# 2. Install Python wrappers and APIs using pip
# pdfplumber: The best library for extracting embedded text and tables from PDFs.
# pytesseract: The Python bridge that talks to the Tesseract OCR engine we installed above.
# pdf2image: Converts PDF files to image objects so Tesseract can read them.
# groq: The official API client to talk to the Vision Language Models later.
!pip install -q pdfplumber pytesseract pdf2image groq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# ==========================================
# CELL 2: DATA ROUTING & INGESTION
# ==========================================
import os
from google.colab import drive, files

# Define a global list that will hold the paths to all our documents
#document_paths = []

# ---------------------------------------------------------
# MECHANISM 1: GOOGLE DRIVE (USED FOR BETA TESTING)
# ---------------------------------------------------------
#print("Mounting Google Drive...")
#drive.mount('/content/drive')

#DRIVE_FOLDER = '/content/drive/MyDrive/Ginja AI case study'

# Loop through the folder and grab every file that is a PDF, JPG, or PNG
#if os.path.exists(DRIVE_FOLDER):
    #for filename in os.listdir(DRIVE_FOLDER):
        #if filename.lower().endswith(('.pdf', '.png', '.jpg', '.jpeg')):
            #full_path = os.path.join(DRIVE_FOLDER, filename)
            #document_paths.append(full_path)
    #print(f"Loaded {len(document_paths)} documents from Google Drive.")
#else:
    #print("WARNING: Drive folder not found. Please check the DRIVE_FOLDER path.")

# ---------------------------------------------------------
# MECHANISM 2: LOCAL UPLOAD (FOR GINJA AI TEAM)
# ---------------------------------------------------------
# To use this, you would uncomment the block below. It opens a standard upload widget.
# print("Please upload your sample documents:")
# uploaded = files.upload()
# for filename in uploaded.keys():
#     document_paths.append(os.path.abspath(filename))
# print(f"Loaded {len(uploaded)} documents via manual upload.")

# ---------------------------------------------------------
# MECHANISM 3: GITHUB CLONE (FOR FINAL SUBMISSION)
 ---------------------------------------------------------
!git clone https://github.com/TIMOMWATA2/data_extraction_claims_pipeline.git

REPO_FOLDER = '/content/data_extraction_claims_pipeline/Ginja_AI_Case_Study_Claims/data/sample_inputs'

import os
for filename in os.listdir(REPO_FOLDER):
    if filename.lower().endswith(('.pdf', '.png', '.jpg', '.jpeg')):
        document_paths.append(os.path.join(REPO_FOLDER, filename))

print(f"Loaded {len(document_paths)} documents directly from GitHub repository.")

Mounting Google Drive...
Mounted at /content/drive
Loaded 6 documents from Google Drive.


In [ ]:
# ==========================================
# CELL 3: FORMAT DETECTION
# ==========================================
import pdfplumber

def detect_document_type(file_path):
    """
    Determines if a file is an image, a text-based PDF, or a scanned PDF.
    Reasoning: Text-based PDFs have embedded text layers. Scanned PDFs are just
    pictures of paper wrapped in a PDF file format, meaning their text layer is empty.
    """
    # 1. Check the file extension first
    ext = file_path.lower().split('.')[-1]
    if ext in ['png', 'jpg', 'jpeg']:
        return "Image (Requires OCR or VLM)"

    # 2. If it's a PDF, we must peek inside to see if it has a text layer
    if ext == 'pdf':
        try:
            # Open the PDF using pdfplumber
            with pdfplumber.open(file_path) as pdf:
                extracted_text = ""
                # Loop through the first few pages (or just the first) to check for text
                for page in pdf.pages:
                    # extract_text() pulls the embedded text. If it's a scan, this returns None or ""
                    text = page.extract_text()
                    if text:
                        extracted_text += text

                # Strip out blank spaces and check if we actually got meaningful text
                if len(extracted_text.strip()) > 10:  # 10 chars is a safe threshold
                    return "Text-based PDF"
                else:
                    return "Scanned PDF (Requires OCR or VLM)"
        except Exception as e:
            return f"Error reading PDF: {e}"

    return "Unknown Format"

# --- TESTING THE PIPELINE SO FAR ---
print("--- FORMAT DETECTION RESULTS ---")
# We loop through the files we ingested in Cell 2 and test our function
for doc_path in document_paths:
    # os.path.basename just gets the file name (e.g., 'claim1.pdf') instead of the massive folder path
    file_name = os.path.basename(doc_path)
    doc_type = detect_document_type(doc_path)
    print(f"File: {file_name} --> Classification: {doc_type}")

--- FORMAT DETECTION RESULTS ---
File: Nairobi_Lifecare_Hospital_Invoice_Wanjiku_Kamau.pdf --> Classification: Text-based PDF
File: Nairobi_Lifecare_Hospital_Invoice_Sharma_Siddharth.pdf --> Classification: Text-based PDF
File: Nairobi_Lifecare_Hospital_Invoice_Deepak_Bodkhe.pdf --> Classification: Text-based PDF
File: Sharma Siddharth Claim Form.pdf --> Classification: Scanned PDF (Requires OCR or VLM)
File: Deepak Bodkhe Claim Form.pdf --> Classification: Scanned PDF (Requires OCR or VLM)
File: claim form Wanjiku Kamua.pdf --> Classification: Scanned PDF (Requires OCR or VLM)


In [ ]:
# ==========================================
# CELL 4: RAW TEXT EXTRACTION (PDF & OCR)
# ==========================================
import cv2
import numpy as np
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

# --- 1. OCR PREPROCESSING (Capability 3 requirement) ---
def preprocess_image_for_ocr(image):
    """
    Cleans up an image before passing it to Tesseract.
    Removes shadows, increases contrast, and sharpens text.
    """
    # Convert PIL image to an OpenCV numpy array
    open_cv_image = np.array(image)

    # Convert to grayscale (OCR doesn't need color, it just adds noise)
    gray = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2GRAY)

    # Apply a threshold to make the text pitch black and the background pure white.
    # We use OTSU's thresholding which calculates the optimal contrast point automatically.
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # (Optional deskewing could go here, but thresholding handles 90% of bad scans)
    return Image.fromarray(binary)

# --- 2. THE EXTRACTION ENGINES ---
def extract_via_pdfplumber(file_path):
    """Extracts embedded text from a digital PDF."""
    full_text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                extracted = page.extract_text()
                if extracted:
                    full_text += extracted + "\n"
        return full_text
    except Exception as e:
        return f"PDF Extraction Error: {e}"

def extract_via_ocr(file_path, is_pdf=False):
    """Converts images/scanned PDFs to text using Tesseract."""
    full_text = ""
    try:
        images = []
        if is_pdf:
            # Convert PDF pages into images (300 DPI is standard for OCR)
            images = convert_from_path(file_path, dpi=300)
        else:
            # It's already an image file (png, jpg)
            images = [Image.open(file_path)]

        # Process each page/image
        for img in images:
            clean_img = preprocess_image_for_ocr(img)
            # Run Tesseract OCR on the cleaned image
            text = pytesseract.image_to_string(clean_img)
            full_text += text + "\n"

        return full_text
    except Exception as e:
        return f"OCR Extraction Error: {e}"

# --- 3. THE ROUTER (Testing the pipeline so far) ---
print("--- RAW TEXT EXTRACTION RESULTS ---")

# We will store our results in a dictionary to start building toward the final JSON output
extraction_results = {}

for doc_path in document_paths:
    file_name = os.path.basename(doc_path)
    doc_type = detect_document_type(doc_path) # From Cell 3

    raw_text = ""
    method_used = ""

    if doc_type == "Text-based PDF":
        raw_text = extract_via_pdfplumber(doc_path)
        method_used = "pdf_text"
    elif doc_type == "Scanned PDF (Requires OCR or VLM)":
        raw_text = extract_via_ocr(doc_path, is_pdf=True)
        method_used = "ocr"
    else:
        raw_text = extract_via_ocr(doc_path, is_pdf=False)
        method_used = "ocr"

    # Save to our dictionary
    extraction_results[file_name] = {
        "format": doc_type,
        "method_used": method_used,
        "raw_text_snippet": raw_text.strip()[:200] # First 200 chars for the audit log
    }

    # Print the snippet to verify it worked
    snippet = extraction_results[file_name]["raw_text_snippet"].replace('\n', ' ')
    print(f"\n[{file_name}] - Method: {method_used.upper()}")
    print(f"Snippet: {snippet}...")

--- RAW TEXT EXTRACTION RESULTS ---

[Nairobi_Lifecare_Hospital_Invoice_Wanjiku_Kamau.pdf] - Method: PDF_TEXT
Snippet: NAIROBI Nairobi Lifecare Hospital LIFECARE Outpatient Invoice Full Name: Wanjiku Kamau Insurance Member No: 1538500 Hospital No: 400120 Visit Type: Outpatient Phone: 250781261474 Date: 12 Feb 2026 Ser...

[Nairobi_Lifecare_Hospital_Invoice_Sharma_Siddharth.pdf] - Method: PDF_TEXT
Snippet: Nairobi Lifecare Hospital NAIROBI LIFECARE Outpatient Invoice Invoice No: NLH-INV-2026-003 Full Name: Sharma Siddharth Insurance Member No: 1536500 Hospital No: 100120 Visit Type: Dental Phone: - Date...

[Nairobi_Lifecare_Hospital_Invoice_Deepak_Bodkhe.pdf] - Method: PDF_TEXT
Snippet: Nairobi Lifecare Hospital NAIROBI LIFECARE Outpatient Invoice Invoice No: NLH-INV-2026-002 Full Name: Deepak Bodkhe Insurance Member No: 1536400 Hospital No: 800120 Visit Type: Outpatient Phone: - Dat...

[Sharma Siddharth Claim Form.pdf] - Method: OCR
Snippet: EDEN CARE  Eden Care Medical Health Insura

In [ ]:
# ==========================================
# CELL 5: VLM EXTRACTION VIA GROQ (UPDATED)
# ==========================================
import os
import json
import base64
from io import BytesIO
from groq import Groq
from google.colab import userdata
from PIL import Image
from pdf2image import convert_from_path

# 1. Initialize Groq Client securely using Colab Secrets
try:
    groq_api_key = userdata.get('Groq_API_KEY')
    client = Groq(api_key=groq_api_key)
except Exception as e:
    print("ERROR: Could not find Groq_API_KEY. Please check the spelling in your Colab Secrets tab.")

# 2. Function to dynamically find an active Vision model (Capability 4 Requirement)
def get_active_vision_model(client):
    """
    Queries Groq for active models.
    Groq recently updated their naming conventions and removed the word 'vision'
    from their new multimodal model IDs, so we must check for known active prefixes.
    """
    # Current active multimodal models on Groq
    target_models = ["llama-4-scout", "qwen3.6-27b"]

    try:
        models = client.models.list()

        for model in models.data:
            model_id = model.id.lower()
            # If they bring back the 'vision' tag, catch it
            if 'vision' in model_id:
                return model.id
            # Otherwise, look for our known multimodal targets
            for target in target_models:
                if target in model_id:
                    return model.id

        return "meta-llama/llama-4-scout-17b-16e-instruct" # Safety fallback
    except Exception as e:
        print(f"Error fetching models: {e}")
        return "meta-llama/llama-4-scout-17b-16e-instruct"

# 3. Helper function to encode our document into a Base64 string for the API
def encode_document_to_base64(file_path):
    """Converts a PDF or Image into a base64 string."""
    ext = file_path.lower().split('.')[-1]

    if ext == 'pdf':
        # Grab only the first page for the VLM to save tokens and time
        images = convert_from_path(file_path, first_page=1, last_page=1, dpi=200)
        img = images[0]
    else:
        img = Image.open(file_path)

    # Convert the PIL Image object to bytes, then base64
    buffered = BytesIO()
    # Save as JPEG to keep payload size small
    img.convert('RGB').save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

# 4. The Core VLM Extraction Function
def extract_via_vlm(file_path, model_name):
    """Sends the document to Groq and requests structured JSON extraction."""
    base64_image = encode_document_to_base64(file_path)

    # The prompt explicitly defines the schema requested in the case study
    prompt_text = """
    Extract the following healthcare claim fields from this document.
    Return ONLY a valid JSON object. Do not include markdown formatting or explanations.
    If a field is missing, set its value to null.

    Required JSON schema:
    {
      "claim_id": "string",
      "member_id": "string",
      "provider_name": "string",
      "provider_id": "string",
      "date_of_service": "string",
      "diagnosis_code": "string",
      "procedure_code": "string",
      "claimed_amount": "string or float",
      "currency": "string",
      "invoice_number": "string"
    }
    """

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_text},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                    ]
                }
            ],
            temperature=0.0, # 0.0 forces the model to be deterministic and factual
            # Enforce JSON output mode (Supported by newer Groq models)
            response_format={"type": "json_object"}
        )
        # Parse the string response back into a Python dictionary
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"error": str(e)}

# --- 5. EXECUTING THE VLM EXTRACTION ---
print("--- VLM EXTRACTION VIA GROQ ---")
active_vision_model = get_active_vision_model(client)
print(f"Dynamically selected model: {active_vision_model}\n")

# We will test this on just ONE document first to ensure the API is working properly
test_doc = document_paths[0]
test_doc_name = os.path.basename(test_doc)

print(f"Testing VLM on: {test_doc_name}...")
vlm_result = extract_via_vlm(test_doc, active_vision_model)

# Pretty-print the JSON dictionary
print(json.dumps(vlm_result, indent=2))

--- VLM EXTRACTION VIA GROQ ---
Dynamically selected model: qwen/qwen3.6-27b

Testing VLM on: Nairobi_Lifecare_Hospital_Invoice_Wanjiku_Kamau.pdf...
{
  "claim_id": null,
  "member_id": "1538500",
  "provider_name": "Nairobi Lifecare Hospital",
  "provider_id": "400120",
  "date_of_service": "12 Feb 2026",
  "diagnosis_code": null,
  "procedure_code": null,
  "claimed_amount": 4300,
  "currency": "KES",
  "invoice_number": null
}


In [ ]:
# ==========================================
# CELL 6: PARSING & VALIDATION ENGINES
# ==========================================
import re
from datetime import datetime
from dateutil import parser

# --- 1. NORMALIZATION (Capability 5) ---
def normalize_extracted_data(raw_dict):
    """
    Cleans up the raw dictionary values returned by our extraction methods.
    Converts dates to ISO 8601 and amounts to strict floats.
    """
    clean_dict = raw_dict.copy()

    # Clean Claimed Amount (e.g., "KES 14,500/-" -> 14500.0)
    if clean_dict.get('claimed_amount'):
        val = str(clean_dict['claimed_amount'])
        # Strip out everything except numbers and decimals
        numeric_str = re.sub(r'[^\d.]', '', val)
        try:
            clean_dict['claimed_amount'] = float(numeric_str)
        except ValueError:
            clean_dict['claimed_amount'] = None

    # Clean Date of Service (e.g., "12 Feb 2026" -> "2026-02-12")
    if clean_dict.get('date_of_service'):
        try:
            # dateutil is incredibly smart at figuring out weird date formats
            parsed_date = parser.parse(clean_dict['date_of_service'], fuzzy=True)
            clean_dict['date_of_service'] = parsed_date.strftime('%Y-%m-%d')
        except Exception:
            # If it completely fails to parse, leave it as is so validation catches it
            pass

    # Clean Codes (Remove spaces and standardize to uppercase)
    for code_field in ['diagnosis_code', 'procedure_code']:
        if clean_dict.get(code_field):
            clean_dict[code_field] = str(clean_dict[code_field]).replace(" ", "").upper()

    return clean_dict

# --- 2. VALIDATION (Capability 6) ---
def validate_record(clean_dict):
    """
    Checks the normalized data against business rules.
    Returns a list of missing fields and a list of validation warning flags.
    """
    missing_fields = []
    validation_flags = []

    # Rule 1: Check Required Fields
    required_fields = ['claim_id', 'member_id', 'claimed_amount', 'date_of_service']
    for field in required_fields:
        # If the key doesn't exist, or the value is None/Empty
        if field not in clean_dict or not clean_dict[field]:
            missing_fields.append(field)

    # Rule 2: Validate Amount Plausibility
    amount = clean_dict.get('claimed_amount')
    if amount is not None:
        if isinstance(amount, float) and amount <= 0:
            validation_flags.append("Amount is zero or negative")
        elif isinstance(amount, float) and amount > 5000000: # Plausibility check
            validation_flags.append("Amount exceeds plausible threshold")

    # Rule 3: Validate Date (No future dates)
    date_str = clean_dict.get('date_of_service')
    if date_str:
        try:
            # Assuming current context is June 2026
            doc_date = datetime.strptime(date_str, '%Y-%m-%d')
            current_date = datetime.now()
            if doc_date > current_date:
                validation_flags.append(f"Future date detected: {date_str}")
        except ValueError:
             validation_flags.append(f"Invalid date format: {date_str}")

    return missing_fields, validation_flags

print("Normalization and Validation functions loaded successfully.")

Normalization and Validation functions loaded successfully.


In [ ]:
# ==========================================
# CELL 7: THE MASTER PIPELINE & EXPORT
# ==========================================
import pandas as pd
import json
import re

def parse_raw_text_to_fields(raw_text):
    """
    Uses Regular Expressions (RegEx) to hunt for specific fields within the raw
    text block dumped by pdfplumber or Tesseract.
    """
    extracted = {
        "claim_id": None, "member_id": None, "provider_name": None,
        "provider_id": None, "date_of_service": None, "diagnosis_code": None,
        "procedure_code": None, "claimed_amount": None, "currency": "KES",
        "invoice_number": None
    }

    # 1. Member ID: Looks for "Member No" or "Membership number" followed by digits
    member_match = re.search(r'(?:Member No|Membership number)[,:\s]*([0-9]{4,})', raw_text, re.IGNORECASE)
    if member_match:
        extracted["member_id"] = member_match.group(1)

    # 2. Invoice / Claim ID: Looks for "Invoice No" or "Visit ID"
    invoice_match = re.search(r'(?:Invoice No|Visit ID)[:\s]*([A-Z0-9\-]+)', raw_text, re.IGNORECASE)
    if invoice_match:
        extracted["invoice_number"] = invoice_match.group(1)
        # We will use the Invoice/Visit ID as the Claim ID for this schema
        extracted["claim_id"] = invoice_match.group(1)

    # 3. Provider Name: Simple keyword matching based on known formats
    if "Nairobi Lifecare" in raw_text:
        extracted["provider_name"] = "Nairobi Lifecare Hospital"
    elif "Eden Care" in raw_text:
        extracted["provider_name"] = "Eden Care Medical"

    # 4. Date of Service: Looks for "Date:" followed by a date pattern
    date_match = re.search(r'Date[:\s]*([\d]{1,2}\s*[A-Za-z]+\s*[\d]{4})', raw_text, re.IGNORECASE)
    if date_match:
        extracted["date_of_service"] = date_match.group(1)

    # 5. Amount: Looks for KES, Amount, or Total followed by numbers
    # Note: RegEx for financial amounts across different formats is notoriously difficult
    amount_match = re.search(r'(?:KES|Amount|Total)[:\s]*([\d,]+(?:\.\d{2})?)', raw_text, re.IGNORECASE)
    if amount_match:
        extracted["claimed_amount"] = amount_match.group(1)

    return extracted

# --- THE CONVEYOR BELT ---
print("🚀 Starting Master Pipeline Processing...\n")

final_records = []

for doc_path in document_paths:
    file_name = os.path.basename(doc_path)

    # 1. Format Detection
    doc_type = detect_document_type(doc_path)
    method_used = ""
    raw_text = ""

    # 2. Routing & Extraction
    if doc_type == "Text-based PDF":
        raw_text = extract_via_pdfplumber(doc_path)
        method_used = "pdf_text"
        # Confidence is high because it's digital text
        confidence_score = 1.0
    else:
        # It's an image or scanned PDF
        is_pdf = doc_path.lower().endswith('.pdf')
        raw_text = extract_via_ocr(doc_path, is_pdf=is_pdf)
        method_used = "ocr"
        # Confidence is lower due to OCR noise
        confidence_score = 0.7

    # 3. Parse the raw text into our schema using RegEx
    raw_fields = parse_raw_text_to_fields(raw_text)

    # 4. Normalize the data (e.g., Dates to ISO 8601, Amounts to floats)
    normalized_fields = normalize_extracted_data(raw_fields)

    # 5. Validate the data and get flags
    missing_fields, validation_flags = validate_record(normalized_fields)

    # 6. Assemble the final structured record required by the case study
    record = {
        "document_name": file_name,
        "claim_id": normalized_fields.get("claim_id"),
        "extraction_method": method_used,
        "confidence_score": confidence_score,
        "fields_extracted": normalized_fields,
        "fields_missing": missing_fields,
        "validation_flags": validation_flags,
        "raw_text_snippet": raw_text.strip()[:200].replace('\n', ' ')
    }

    final_records.append(record)

# --- EXPORTING THE RESULTS ---
# Convert the list of dictionaries into a Pandas DataFrame for beautiful visualization and easy export
df_results = pd.DataFrame(final_records)

# Save to CSV (Deliverable Requirement)
csv_filename = "structured_claims_output.csv"
df_results.to_csv(csv_filename, index=False)

# Save to JSON (Deliverable Requirement)
json_filename = "structured_claims_output.json"
with open(json_filename, 'w') as f:
    json.dump(final_records, f, indent=4)

print(f"✅ Pipeline complete! Processed {len(document_paths)} documents.")
print(f"📁 Results saved to: {csv_filename} and {json_filename}\n")

# Display the DataFrame cleanly in Colab
# We drop the massive 'fields_extracted' dictionary from the visual display just to keep the table readable
display(df_results.drop(columns=['fields_extracted', 'raw_text_snippet']))

🚀 Starting Master Pipeline Processing...

✅ Pipeline complete! Processed 6 documents.
📁 Results saved to: structured_claims_output.csv and structured_claims_output.json



,document_name,claim_id,extraction_method,confidence_score,fields_missing,validation_flags
0,Nairobi_Lifecare_Hospital_Invoice_Wanjiku_Kama...,None,pdf_text,1.0,[claim_id],[]
1,Nairobi_Lifecare_Hospital_Invoice_Sharma_Siddh...,NLH-INV-2026-003,pdf_text,1.0,[],[]
2,Nairobi_Lifecare_Hospital_Invoice_Deepak_Bodkh...,NLH-INV-2026-002,pdf_text,1.0,[],[]
3,Sharma Siddharth Claim Form.pdf,20011942,ocr,0.7,"[claimed_amount, date_of_service]",[]
4,Deepak Bodkhe Claim Form.pdf,20011941,ocr,0.7,"[claimed_amount, date_of_service]",[]
5,claim form Wanjiku Kamua.pdf,20011943,ocr,0.7,"[claimed_amount, date_of_service]",[]
